# Predicción del precio de autos
Base de datos de [Rehan Liaqat en Kaggle](https://www.kaggle.com/datasets/rehan497/car-price-prediction-dataset)

Análisis por Luis Gerardo Ramirez Archundia | 🇲🇽


# Librerías
En esta sección se importarán las librerías necesarias paraeste proyecto.

In [1]:
# librerías básicas
import pandas as pd
import os
import sys
import numpy as np

# visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# versiones utilizadas
print(f'Versión de Python {sys.version}')
print(f'Versión de Pandas {pd.__version__}')

Versión de Python 3.10.13 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:15:57) [MSC v.1916 64 bit (AMD64)]
Versión de Pandas 2.3.2


# Datos

In [2]:
df=pd.read_csv('data/car_price_prediction_.csv')
print('Los datos fueron importados correctamente!')
# motramos parte de los datos
df.head(5)

Los datos fueron importados correctamente!


,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,4,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


In [3]:
# infomración general de los datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Car ID        2500 non-null   int64  
 1   Brand         2500 non-null   object 
 2   Year          2500 non-null   int64  
 3   Engine Size   2500 non-null   float64
 4   Fuel Type     2500 non-null   object 
 5   Transmission  2500 non-null   object 
 6   Mileage       2500 non-null   int64  
 7   Condition     2500 non-null   object 
 8   Price         2500 non-null   float64
 9   Model         2500 non-null   object 
dtypes: float64(2), int64(3), object(5)
memory usage: 195.4+ KB


In [4]:
shape=df.shape
print(f"Se tienen {shape[0]} registros y {shape[1]} columnas")

Se tienen 2500 registros y 10 columnas


# Preparación de los datos

In [5]:
# verificamos la existencia de datos nulos
print('Existencia de datos tipo NULL\n')
null_data=df.isnull().sum()
print(null_data)
if null_data.sum()==0:
    print('-----\nNo existen datos tipo NULL')
else:
    print('Se ecnontraron datos tipo NULL')

Existencia de datos tipo NULL

Car ID          0
Brand           0
Year            0
Engine Size     0
Fuel Type       0
Transmission    0
Mileage         0
Condition       0
Price           0
Model           0
dtype: int64
-----
No existen datos tipo NULL


In [6]:
'''
    Algunos de los nombres de las columnas contienen espacios y 
    letras mayúsculas, para evitar complicaciones en el tratamiento
    uniremos las palabras por guiones bajos y todo en minúsculas.
'''
df.columns=df.columns.str.replace(' ', '_')
df.columns=df.columns.str.lower()

print(df.columns)

Index(['car_id', 'brand', 'year', 'engine_size', 'fuel_type', 'transmission',
       'mileage', 'condition', 'price', 'model'],
      dtype='object')


## Inconsistencias de los datos
Se han detectado algunas inconsistencias en los datos:

- Autos eléctricos declaran capacidad del motor en litros
- Autos nuevos declaran kilometraje
- Autos electricos declaran tener transmision manual, todos los autos eléctricos son automaticos
- Autos eléctricos declaran uso de combustible

In [7]:
# pasamos a tipo eléctrico los autos Tesla
df.loc[df['brand']=='Tesla', 'fuel_type']='Electric'
# cambiamos la capacidad del motor a pd.Na
df.loc[df['brand']=='Tesla', 'engine_size']=np.nan
# cambiamos el tipo de transmisión a automática a todos los Tesla
df.loc[df['brand']=='Tesla', 'transmission']='Automatic'
# en autos electricos no existen las transmisiones manuales
df.loc[df['fuel_type']=='Electric', 'transmission']='Automatic'
# para autos nuevos el kilometraje debe ser cero
df.loc[df['condition']=='New', 'mileage']=0
df.head(5)


,car_id,brand,year,engine_size,fuel_type,transmission,mileage,condition,price,model
0,1,Tesla,2016,NaN,Electric,Automatic,0,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Automatic,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Automatic,0,New,44402.61,A4
3,4,Tesla,2011,NaN,Electric,Automatic,0,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


# Análisis exploratorio

## ¿Cuál es el rango general de precios?


In [8]:
price=df['price']
fig=px.violin(
    data_frame=price,
    box=True,
    points='outliers',
    title='Rango general de precios'
)
fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(
        gridcolor='lightgrey',
        title='Precio ($)'
    ),
    xaxis=dict(
        title=None
    )
)
fig.show()

In [9]:
df['price'].describe()

count     2500.000000
mean     52638.022532
std      27295.833455
min       5011.270000
25%      28908.485000
50%      53485.240000
75%      75838.532500
max      99982.590000
Name: price, dtype: float64

**Observaciones:**
- El rango de precios va desde \$5,011 hasta \$99,982.
- El gráfico de caja no muestra valores atípicos en los precios.
- El auto típico (mediana) tiene un precio de aproximadamante \$53,485.24.
- El gráfico de violín muestra una distribución ancha desde los \$20,000 hasta aproximadamente \$90,000

## Rango de precios por marca

In [10]:
precio_autos=df[['brand', 'condition', 'price']]

In [26]:
fig=px.violin(
    data_frame=precio_autos,
    y='price',
    color='brand',
    facet_row='condition',
    box=True,
    title='Distribución de precios por marca y condición'
)
fig.update_layout(
    plot_bgcolor='white',
    legend_title='Marca',
    height=800,
    width=1000,
    margin=dict(t=80, b=40, l=40, r=40)
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgrey",
    gridwidth=0.5,
    griddash="dot",
    zeroline=True,
    zerolinecolor='lightgrey',
    dtick=20000
)
fig.for_each_annotation(
    lambda a: a.update(
        text=a.text.split("=")[-1]
    )
)
fig.show()

In [12]:
precio_autos.groupby(by=['brand', 'condition']).describe()

price                                                   \
                    count          mean           std      min         25%   
brand    condition                                                           
Audi     Like New   111.0  51931.677658  27072.727329  5703.33  30143.8800   
         New        129.0  50789.721473  28410.137674  5022.86  28199.4200   
         Used       128.0  53145.078437  26770.808353  5011.27  31256.6475   
BMW      Like New   138.0  54000.022319  28132.634118  5124.89  26696.4625   
         New        104.0  54920.846538  28147.634770  5741.64  32905.2575   
         Used       116.0  53659.274397  26324.766604  5107.22  31205.2550   
Ford     Like New   124.0  52967.725403  27822.603383  5865.44  29980.1350   
         New        113.0  53635.520088  27021.503182  5247.71  30419.3000   
         Used       110.0  47945.888182  25912.931201  5537.99  25464.3050   
Honda    Like New   123.0  52429.640163  27552.923590  5060.75  27809.6550   
         New        109.0  53297.152477  25034.618994  6482.71  31373.1900   
         Used       120.0  50528.871583  25294.985034  5535.30  29917.0900   
Mercedes Like New   110.0  53181.418727  27969.831724  5931.77  30404.3425   
         New        115.0  52781.500348  28688.079577  6493.08  27310.6200   
         Used       128.0  53567.392187  28587.339520  5353.03  29094.0050   
Tesla    Like New   121.0  55098.004793  28784.804277  5343.23  28996.5300   
         New        119.0  49380.045714  26425.997772  5558.56  25461.2850   
         Used       108.0  56170.430556  28311.191304  7292.61  29907.1475   
Toyota   Like New   109.0  54968.755413  26300.940212  5472.39  35411.8800   
         New        120.0  49256.708917  28072.397811  6729.50  22482.1550   
         Used       145.0  52241.689310  26559.465801  6178.92  32071.9900   

                                                     
                          50%         75%       max  
brand    condition                                   
Audi     Like New   46849.570  72255.0450  98153.00  
         New        46049.780  72440.1300  99982.59  
         Used       52992.345  72218.2400  98434.45  
BMW      Like New   56784.665  80161.0000  98972.03  
         New        58779.595  75919.0400  97684.22  
         Used       56810.935  70536.9475  99968.62  
Ford     Like New   54961.225  77860.9775  97090.31  
         New        58132.970  76481.9900  99605.33  
         Used       43716.495  69007.0150  99153.11  
Honda    Like New   56401.450  74437.1150  99500.97  
         New        51706.440  75094.3100  99578.74  
         Used       50719.930  70662.1250  97386.03  
Mercedes Like New   55132.835  77581.4275  98190.42  
         New        51393.690  79383.6000  99754.42  
         Used       54276.245  78747.5650  99072.60  
Tesla    Like New   56797.720  82465.8000  99905.90  
         New        49283.410  70373.7350  98704.48  
         Used       58360.320  83332.8300  99794.46  
Toyota   Like New   55114.420  76065.1400  98493.27  
         New        50203.075  74130.6325  98645.75  
         Used       50250.980  74744.2100  99400.47

**Lo que se muestra en la gráfica:**

- BMW muestra una distribución de probabilidad con un abultamiento por encima de los \$60k, lo que indica que la mayoría de los autos que ofrece esta marca están concentrados en dicha región.
- Audi presenta el caso opuesto, el abultamiento empieza por debajo de los \$50k por lo que respecto a BMW, Audi ofrece autos más económicos.
- Ford, Honda, Mercedes y Toyota tienen un comportamiento similar en el gráfico de violín, lo que parece indicar una segmentación de los autos nuevos en dos gamas: premium y accesible.

## ¿Qué tan recorridos están los autos? 

In [13]:
mileage=df['mileage']

fig=px.histogram(
    data_frame=mileage,
    title='Histograma de millaje'
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(
        title='Millas'
    ),
    yaxis=dict(
        gridcolor='lightgrey',
        title='Conteo'
    ),
    legend_title=None
)
fig.show()

**Lo que se muestra en la gráfica:**

En el rango de 0 a 10k millas se concentra la mayoría de autos, llegando hasta 863. A partir de ahí no se observan grupos relevantes de autos que tengan determinado número de millas. Este dato puede tomarse como un valor atípico.

### Rango de 0 a 10k millas
Conviene desglosar el rango de 0 a 10k millas:

In [14]:
mileage_filtrado=df[df['mileage']<=10000]
fig=px.histogram(
    data_frame=mileage_filtrado,
    x='mileage',
    log_y=True,
    opacity=0.8,
    nbins=50,
    title='Histograma de millaje para el rango de 0 a 10k millas'
)
fig.update_traces(
    marker_line_color='black', 
    marker_line_width=1,
    texttemplate='%{y}',
    textposition='outside'
    )
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(
        title='Millaje'
    ),
    yaxis=dict(
        title='Conteo (escala log)',
        gridcolor='lightgrey'
    )
)
fig.show()

**Lo que se muestra en la gráfica:**

Existen 812 autos que tienen entre 0 y 812 millas recorridas.

## ¿Qué tipo de autos ofrecen las marcas?

In [15]:
'''
    Veamos cuales son las marcas de autos y el tipo de combustible que utilizan
'''
tipos_y_marcas=df.groupby(by=['brand'])['fuel_type'].value_counts().reset_index()

In [16]:
fig=px.bar(
    data_frame=tipos_y_marcas,
    x='brand',
    y='count',
    color='fuel_type',
    barmode='group',
    title='Tipo de combustible por marca'
)
fig.update_layout(
    legend=dict(
        title="Tipo de combustible"
    ),
    xaxis=dict(
        title='Marca'
    ),
    yaxis=dict(
        title='Conteo',
        gridcolor='lightgrey'

    ),
    plot_bgcolor='white'
)
fig.show()

**Lo que se muestra en la gráfica:**

Tesla es la única marca que solo ofrece autos eléctricos.

## ¿Qué tipos de transmisiones se ofrecen?

In [17]:
marcas_y_transmisiones=df.groupby(by=['brand'])['transmission'].value_counts().reset_index()

In [18]:
fig=px.bar(
    data_frame=marcas_y_transmisiones,
    x='brand',
    y='count',
    color='transmission',
    title='Transmisiones de cada marca',
    barmode='group'
)
fig.update_layout(
    plot_bgcolor='white',
    legend_title='Transmisión',
    xaxis=dict(
        title='Marca'
    ),
    yaxis=dict(
        gridcolor='lightgrey'
    )
)
fig.show()

**Lo que se muestra en la gráfica:**

- Al ser autos eléctricos, Tesla solo maneja transmisiones automáticas.
- En todas las marcas existe una preferencia muy marcada por ofrecer autos de caja automática.

## ¿Cuál es la condición de los vehículos?

In [19]:
marca_y_condicion=df.groupby(by=['brand'])['condition'].value_counts().reset_index()

In [20]:
fig=px.bar(
    data_frame=marca_y_condicion,
    x='brand',
    y='count',
    color='condition',
    title='Condición de los vehículos por marca.',
    barmode='group'
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(
        title='Marca'
    ),
    yaxis=dict(
        title='Conteo',
        gridcolor='lightgrey'
    ),
    legend_title='Condición'
)
fig.show()

**Lo que se muestra en la gráfica:**

- Toyota es la marca que más vehículos vehículos usados ofrece.
- Ninguna marca destaca por ofrecer más vehículos nuevos.
- BMW tiene una precencia considerable en los seminuevos.